In [1]:
%pip install -q pinecone sentence-transformers numpy python-dotenv tf-keras

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Week 5 Tuesday -- Embeddings, Similarity & Pinecone Foundations

## Overview

Last week we built chains and agents with **LangChain** and observed them with **LangSmith**. But every intelligent retrieval system needs one critical ingredient: **data the model can search through at query time**. That ingredient lives inside a **vector database**.

Today we answer three questions:
1. What *are* vectors and embeddings, and why do they matter for AI?
2. How do we measure whether two pieces of text are "similar"?
3. How do we store, index, and query embeddings at scale with **Pinecone**?

By the end of this session you will understand the foundational layer that powers **Retrieval-Augmented Generation (RAG)** — the pattern that lets an LLM answer questions about *your* data instead of hallucinating.

---

## Learning Objectives

- Define vectors and embeddings; explain how they capture semantic meaning
- Compare similarity metrics: cosine similarity, Euclidean distance, dot product
- Set up Pinecone, create an index, and perform upsert / query operations
- Understand CRUD patterns in a cloud vector database

---

## Roadmap

| Stage | Topic | Focus |
|-------|-------|-------|
| **1** | Vector Intuition & Embeddings | Conceptual foundation with simple numpy / sentence-transformers code |
| **2** | Similarity Metrics | Cosine, Euclidean, dot product — building intuition with code |
| **3** | Pinecone Setup & Index Operations | Quickstart, CRUD with Pinecone |

---

# Stage 1 — Vector Intuition & Embeddings

## What is a vector?

A **vector** is just an ordered list of numbers — coordinates in some space.

```
2-D vector:  [3, 4]          → a point on a flat plane
384-D vector: [0.23, -0.17, 0.89, ..., -0.45]  → a point in embedding space
```

Traditional databases store the *words* in a document. Vector databases store the *meaning* of a document as a point in space.

## What is an embedding?

An **embedding** is a vector produced by a trained neural network that captures the *semantic meaning* of the input. The magic property:

```
"I love dogs"    → [0.82, 0.45, -0.12, ...]
"I adore puppies" → [0.79, 0.48, -0.15, ...]   ← very similar vectors!
"I hate Mondays"  → [-0.34, 0.11, 0.67, ...]   ← very different vector
```

Even though the *words* differ, the *meanings* are close — and so are the vectors. This is the insight that makes semantic search, recommendations, and RAG possible.

## Why dimensions matter

| Dimensions | Example Model | Trade-off |
|-----------|---------------|----------|
| 384 | `all-MiniLM-L6-v2` | Fast, lightweight, good for learning |
| 768 | `all-mpnet-base-v2` | Higher quality, more storage |
| 1024 | Pinecone `llama-text-embed-v2` | Integrated embedding on Pinecone |
| 1536 | OpenAI `text-embedding-3-small` | Production quality, API cost |

Think of dimensions like image resolution: more dimensions capture more nuance but cost more to store and compare.

In [5]:
import numpy as np

# A vector is just a list of numbers
v1 = np.array([3, 4])
v2 = np.array([6, 8])

print(f"v1 = {v1}")
print(f"v2 = {v2}")
print(f"v1 magnitude (length): {np.linalg.norm(v1):.2f}")
print(f"v2 magnitude (length): {np.linalg.norm(v2):.2f}")
print(f"v2 is v1 scaled by 2 — same direction, different magnitude")

v1 = [3 4]
v2 = [6 8]
v1 magnitude (length): 5.00
v2 magnitude (length): 10.00
v2 is v1 scaled by 2 — same direction, different magnitude


In [16]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "The quick brown fox jumps over the lazy dog",
    "A fast auburn fox leaps above a sleepy canine",
    "Machine learning is transforming industries",
]

embeddings = model.encode(sentences)

print(f"Number of sentences: {len(embeddings)}")
print(f"Dimensions per embedding: {embeddings.shape[1]}")
print(f"\nFirst 10 values of sentence 0: {embeddings[0][:10].round(4)}")


Number of sentences: 3
Dimensions per embedding: 384

First 10 values of sentence 0: [ 0.0355  0.0613  0.0527  0.0707  0.0331 -0.0307  0.0066 -0.0612 -0.0013
  0.0106]


Each sentence is now a 384-dimensional vector. Sentences with similar meaning should be *close* in this space. How do we measure "close"? That brings us to Stage 2.

---

# Stage 2 — Similarity Metrics

Every vector-database query boils down to one question: **how close are these two vectors?** There are three common ways to answer that.

## Cosine Similarity

Measures the *angle* between two vectors, ignoring magnitude.

$$\text{cosine\_sim}(A, B) = \frac{A \cdot B}{\|A\| \, \|B\|}$$

| Value | Meaning |
|-------|--------|
| 1.0 | Identical direction |
| 0.0 | Perpendicular / unrelated |
| -1.0 | Opposite direction |

Best for **text** — direction matters more than magnitude.

## Euclidean Distance (L2)

Straight-line distance between two points.

$$d(A, B) = \sqrt{\sum_i (A_i - B_i)^2}$$

Lower = more similar. Sensitive to magnitude.

## Dot Product

$$A \cdot B = \sum_i A_i \cdot B_i$$

When vectors are **normalized** (unit length), dot product equals cosine similarity — and it's the fastest to compute.

### Which to choose?

| Metric | Best For | Notes |
|--------|----------|-------|
| Cosine | Text embeddings | Direction matters, magnitude doesn't |
| Euclidean | Images, dense vectors | Absolute position matters |
| Dot Product | Normalized vectors | Fastest computation |

In [6]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def euclidean_distance(a, b):
    return np.linalg.norm(a - b)

def dot_product(a, b):
    return np.dot(a, b)

print("Comparing the three fox/ML sentences we embedded above:\n")

pairs = [
    (0, 1, "Fox sentence vs. Fox paraphrase"),
    (0, 2, "Fox sentence vs. ML sentence"),
]

for i, j, label in pairs:
    cos  = cosine_similarity(embeddings[i], embeddings[j])
    euc  = euclidean_distance(embeddings[i], embeddings[j])
    dot_ = dot_product(embeddings[i], embeddings[j])
    print(f"{label}")
    print(f"  Cosine similarity : {cos:.4f}")
    print(f"  Euclidean distance: {euc:.4f}")
    print(f"  Dot product       : {dot_:.4f}")
    print()

Comparing the three fox/ML sentences we embedded above:

Fox sentence vs. Fox paraphrase
  Cosine similarity : 0.7333
  Euclidean distance: 0.7303
  Dot product       : 0.7333

Fox sentence vs. ML sentence
  Cosine similarity : -0.0419
  Euclidean distance: 1.4435
  Dot product       : -0.0419



Notice how the two fox sentences have **high cosine similarity** and **low Euclidean distance**, while the fox-vs-ML pair is the opposite. Cosine similarity is the default in Pinecone for text workloads, and it's what we'll use going forward.

In [7]:
# Why cosine beats Euclidean for text: magnitude independence

short = np.array([1, 1])
long  = np.array([10, 10])

print("Same direction, different magnitudes:")
print(f"  Euclidean distance: {euclidean_distance(short, long):.2f}  ← looks very different!")
print(f"  Cosine similarity : {cosine_similarity(short, long):.2f}  ← identical direction")

Same direction, different magnitudes:
  Euclidean distance: 12.73  ← looks very different!
  Cosine similarity : 1.00  ← identical direction


In [13]:
# A more realistic example: semantic search over a tiny "database"

documents = [
    "How to train a machine learning model",
    "Introduction to neural networks",
    "Best pizza recipes for beginners",
    "Understanding deep learning concepts",
    "Italian cooking techniques",
    "Python programming tutorial",
]

doc_embeddings = model.encode(documents)

query = "Cook with Italian cooking techniques"
query_embedding = model.encode(query)

print(f"Query: '{query}'\n")

scored = []
for doc, emb in zip(documents, doc_embeddings):
    sim = cosine_similarity(query_embedding, emb)
    scored.append((sim, doc))

scored.sort(reverse=True)
for sim, doc in scored:
    tag = "Relevant" if sim > 0.40 else "Not relevant"
    print(f"  [{sim:.3f}] {tag:>12} | {doc}")

Query: 'Cook with Italian cooking techniques'

  [0.944]     Relevant | Italian cooking techniques
  [0.463]     Relevant | Best pizza recipes for beginners
  [0.128] Not relevant | Python programming tutorial
  [0.109] Not relevant | How to train a machine learning model
  [0.109] Not relevant | Introduction to neural networks
  [0.006] Not relevant | Understanding deep learning concepts


The AI-related documents float to the top even though the query never mentions "machine learning" or "neural networks" — the embeddings captured the *meaning*, not just the keywords.

### Similarity thresholds (rules of thumb)

| Cosine Similarity | Interpretation |
|-------------------|---------------|
| 0.95 – 1.00 | Near-duplicate / paraphrase |
| 0.80 – 0.95 | Highly related |
| 0.60 – 0.80 | Same topic |
| 0.40 – 0.60 | Loosely related |
| < 0.40 | Different topics |

These vary by model and domain — always calibrate with your own data.

---

# Stage 3 — Pinecone Setup & Index Operations

We've been computing similarity by brute force — comparing the query to *every* vector. That's fine for 6 documents but falls apart at 1 million. **Pinecone** is a managed, cloud-hosted vector database that:

- Stores vectors and metadata
- Builds **ANN (Approximate Nearest Neighbor)** indexes so queries run in O(log n), not O(n)
- Auto-scales — no infrastructure to manage
- Supports serverless billing (pay per use)

### Pinecone core concepts

| Concept | Pinecone Term | SQL Analogy |
|---------|--------------|------------|
| Database | Index | Database |
| Partition | Namespace | Schema / Table |
| Row | Vector (id + values + metadata) | Row |
| Search | `query()` / `search()` | SELECT … ORDER BY similarity |
| Insert/Update | `upsert()` / `upsert_records()` | INSERT … ON CONFLICT UPDATE |

### Two index flavours

1. **Standard index** — you provide your own embedding vectors (`upsert` with `values`).
2. **Integrated-embedding index** — Pinecone embeds text for you at ingest *and* query time (uses `create_index_for_model` + `upsert_records` + `search`).

We'll demonstrate **both** patterns below.

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
if not PINECONE_API_KEY:
    print("WARNING: PINECONE_API_KEY not found in environment.")
    print("Set it in a .env file or export it before running Pinecone cells.")
    
else:
    print("Pinecone API key loaded successfully.")
    print(PINECONE_API_KEY)

Pinecone API key loaded successfully.
pcsk_MrjEH_2B3AV7k71EGa48XwUdmbqLzGxcr9a2aQrxuj319TpZYsM6Vf2MHzww9U1qK3Brq


## 3-A: Integrated-Embedding Index (Pinecone embeds for you)

This is the fastest path to a working semantic search — you send plain text, Pinecone embeds and indexes it.

In [5]:
from pinecone import Pinecone
import time

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "w5-tuesday-quickstart"

if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model": "llama-text-embed-v2",
            "field_map": {"text": "chunk_text"},
        },
    )
    print(f"Creating index '{index_name}'...")
    time.sleep(5)
else:
    print(f"Index '{index_name}' already exists.")

print("Index ready.")

Creating index 'w5-tuesday-quickstart'...
Index ready.


In [6]:
records = [
    {"_id": "rec1",  "chunk_text": "The Eiffel Tower was completed in 1889 and stands in Paris, France.", "category": "history"},
    {"_id": "rec2",  "chunk_text": "Photosynthesis allows plants to convert sunlight into energy.", "category": "science"},
    {"_id": "rec3",  "chunk_text": "Albert Einstein developed the theory of relativity.", "category": "science"},
    {"_id": "rec4",  "chunk_text": "The mitochondrion is often called the powerhouse of the cell.", "category": "biology"},
    {"_id": "rec5",  "chunk_text": "Shakespeare wrote many famous plays, including Hamlet and Macbeth.", "category": "literature"},
    {"_id": "rec6",  "chunk_text": "Water boils at 100°C under standard atmospheric pressure.", "category": "physics"},
    {"_id": "rec7",  "chunk_text": "The Great Wall of China was built to protect against invasions.", "category": "history"},
    {"_id": "rec8",  "chunk_text": "Honey never spoils due to its low moisture content and acidity.", "category": "food science"},
    {"_id": "rec9",  "chunk_text": "The speed of light in a vacuum is approximately 299,792 km/s.", "category": "physics"},
    {"_id": "rec10", "chunk_text": "Newton's laws describe the motion of objects.", "category": "physics"},
]

dense_index = pc.Index(index_name)

dense_index.upsert_records("example-namespace", records)
print(f"Upserted {len(records)} records. Waiting for indexing...")

time.sleep(10)

stats = dense_index.describe_index_stats()
print(f"\nIndex stats: {stats}")

C:\Users\CharlesJester\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Upserted 10 records. Waiting for indexing...

Index stats: {'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '190',
                                    'content-type': 'application/json',
                                    'date': 'Tue, 21 Apr 2026 16:10:38 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '3',
                                    'x-pinecone-request-latency-ms': '2',
                                    'x-pinecone-response-duration-ms': '3'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'example-namespace': {'vector_count': 10}},
 'storageFullness': 0.0,
 'total_vector_count': 10,
 'vector_type': 'dense'}


In [11]:
query = "historical structures and monuments"

results = dense_index.search(
    namespace="example-namespace",
    query={
        "top_k": 5,
        "inputs": {"text": query},
    },
)

print(f"Query: '{query}'\n")
for hit in results["result"]["hits"]:
    print(
        f"  id: {hit['_id']:<6} | "
        f"score: {hit['_score']:.4f} | "
        f"category: {hit['fields']['category']:<12} | "
        f"{hit['fields']['chunk_text']}"
    )

Query: 'historical structures and monuments'

  id: rec7   | score: 0.0730 | category: history      | The Great Wall of China was built to protect against invasions.
  id: rec5   | score: 0.0673 | category: literature   | Shakespeare wrote many famous plays, including Hamlet and Macbeth.
  id: rec1   | score: 0.0513 | category: history      | The Eiffel Tower was completed in 1889 and stands in Paris, France.
  id: rec10  | score: 0.0511 | category: physics      | Newton's laws describe the motion of objects.
  id: rec3   | score: 0.0341 | category: science      | Albert Einstein developed the theory of relativity.


Notice that the history records about the Eiffel Tower, Great Wall, etc. rank highest even though the query never mentions those names — semantic search in action.

### Reranking

Pinecone can optionally **rerank** results with a cross-encoder model for higher relevance. This is a two-stage retrieve-then-rerank pattern:

In [12]:
reranked = dense_index.search(
    namespace="example-namespace",
    query={
        "top_k": 5,
        "inputs": {"text": query},
    },
    rerank={
        "model": "bge-reranker-v2-m3",
        "top_n": 5,
        "rank_fields": ["chunk_text"],
    },
)

print(f"Reranked results for: '{query}'\n")
for hit in reranked["result"]["hits"]:
    print(
        f"  id: {hit['_id']:<6} | "
        f"score: {hit['_score']:.4f} | "
        f"{hit['fields']['chunk_text']}"
    )

Reranked results for: 'historical structures and monuments'

  id: rec1   | score: 0.1122 | The Eiffel Tower was completed in 1889 and stands in Paris, France.
  id: rec7   | score: 0.0925 | The Great Wall of China was built to protect against invasions.
  id: rec5   | score: 0.0000 | Shakespeare wrote many famous plays, including Hamlet and Macbeth.
  id: rec10  | score: 0.0000 | Newton's laws describe the motion of objects.
  id: rec3   | score: 0.0000 | Albert Einstein developed the theory of relativity.


## 3-B: Standard Index (bring your own embeddings)

When you want control over the embedding model (e.g. using `sentence-transformers` locally, or OpenAI's API), you create a standard index and provide the vectors yourself.

In [17]:
from pinecone import ServerlessSpec

std_index_name = "w5-tuesday-standard"
DIMENSION = 384  # all-MiniLM-L6-v2

if not pc.has_index(std_index_name):
    pc.create_index(
        name=std_index_name,
        dimension=DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Creating standard index '{std_index_name}'...")
    while not pc.describe_index(std_index_name).status.ready:
        time.sleep(1)
    print("Index is ready!")
else:
    print(f"Index '{std_index_name}' already exists.")

std_index = pc.Index(std_index_name)

Index 'w5-tuesday-standard' already exists.


In [18]:
# Upsert: insert-or-update vectors

texts = [
    "Machine learning fundamentals and algorithms",
    "Deep learning with neural networks",
    "Natural language processing basics",
    "Computer vision and image recognition",
    "Best pasta recipes from Italy",
]

text_embeddings = model.encode(texts).tolist()

vectors = []
for i, (text, emb) in enumerate(zip(texts, text_embeddings)):
    vectors.append({
        "id": f"doc-{i}",
        "values": emb,
        "metadata": {"text": text, "category": "ai" if i < 4 else "food"},
    })

std_index.upsert(vectors=vectors, namespace="demo")
print(f"Upserted {len(vectors)} vectors.")

time.sleep(5)
print(std_index.describe_index_stats())

Upserted 5 vectors.
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '174',
                                    'content-type': 'application/json',
                                    'date': 'Tue, 21 Apr 2026 16:21:37 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '47',
                                    'x-pinecone-request-latency-ms': '47',
                                    'x-pinecone-response-duration-ms': '48'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'demo': {'vector_count': 5}},
 'storageFullness': 0.0,
 'total_vector_count': 5,
 'vector_type': 'dense'}


In [24]:
# Query with your own embedding

query_text = "How do neural networks learn?"
query_vec = model.encode(query_text).tolist()

results = std_index.query(
    vector=query_vec,
    top_k=5,
    namespace="demo",
    include_metadata=True,
)

print(f"Query: '{query_text}'\n")
for match in results.matches:
    print(
        f"  id: {match.id} | "
        f"score: {match.score:.4f} | "
        f"text: {match.metadata['text']}"
    )

Query: 'How do neural networks learn?'

  id: doc-1 | score: 0.4916 | text: Deep learning with neural networks
  id: doc-0 | score: 0.4547 | text: Machine learning fundamentals and algorithms
  id: doc-3 | score: 0.3274 | text: Computer vision and image recognition
  id: doc-2 | score: 0.2977 | text: Natural language processing basics
  id: doc-4 | score: 0.1215 | text: Best pasta recipes from Italy


## 3-C: CRUD Operations in Pinecone

| Operation | Method | Notes |
|-----------|--------|-------|
| **Create / Update** | `upsert()` | Insert-or-update by ID |
| **Read (by ID)** | `fetch()` | Retrieve specific vectors |
| **Read (similarity)** | `query()` | Nearest-neighbor search |
| **Delete** | `delete()` | Remove by ID or filter |

In [ ]:
# FETCH — retrieve vectors by ID

fetched = std_index.fetch(ids=["doc-0", "doc-4"], namespace="demo")

for vid, vec in fetched.vectors.items():
    print(f"ID: {vid}")
    print(f"  metadata: {vec.metadata}")
    print(f"  values (first 5): {vec.values[:5]}")
    print()

In [ ]:
# UPDATE — upsert with the same ID overwrites the vector

updated_text = "Advanced machine learning: gradient boosting and ensemble methods"
updated_emb = model.encode(updated_text).tolist()

std_index.upsert(
    vectors=[{
        "id": "doc-0",
        "values": updated_emb,
        "metadata": {"text": updated_text, "category": "ai"},
    }],
    namespace="demo",
)

print(f"Updated doc-0 to: '{updated_text}'")

time.sleep(3)
fetched = std_index.fetch(ids=["doc-0"], namespace="demo")
print(f"Verified metadata: {fetched.vectors['doc-0'].metadata}")

In [ ]:
# DELETE — remove vectors

print(f"Before delete: {std_index.describe_index_stats()}")

std_index.delete(ids=["doc-4"], namespace="demo")
print("\nDeleted doc-4 (pasta recipe)")

time.sleep(3)
print(f"\nAfter delete: {std_index.describe_index_stats()}")

In [ ]:
# QUERY WITH METADATA FILTER — combine semantic search with structured filters

query_text = "deep learning techniques"
query_vec = model.encode(query_text).tolist()

results = std_index.query(
    vector=query_vec,
    top_k=5,
    namespace="demo",
    filter={"category": {"$eq": "ai"}},
    include_metadata=True,
)

print(f"Query: '{query_text}' (filtered to category='ai')\n")
for match in results.matches:
    print(f"  {match.id} | score: {match.score:.4f} | {match.metadata['text']}")

## 3-D: Batch Upsert Pattern

In production you'll often upsert thousands or millions of vectors. Pinecone's `upsert` method accepts batches — always prefer batching over single-vector inserts for 10x+ throughput.

In [ ]:
batch_texts = [
    "Reinforcement learning trains agents through rewards",
    "Transformers use self-attention mechanisms",
    "Convolutional neural networks excel at image tasks",
    "Recurrent neural networks handle sequential data",
    "Generative adversarial networks create synthetic data",
    "Transfer learning reuses pretrained model weights",
    "Dimensionality reduction simplifies high-dimensional data",
    "Feature engineering improves model performance",
]

batch_embeddings = model.encode(batch_texts).tolist()

BATCH_SIZE = 100
batch_vectors = [
    {
        "id": f"batch-{i}",
        "values": emb,
        "metadata": {"text": text, "category": "ai"},
    }
    for i, (text, emb) in enumerate(zip(batch_texts, batch_embeddings))
]

for i in range(0, len(batch_vectors), BATCH_SIZE):
    chunk = batch_vectors[i : i + BATCH_SIZE]
    std_index.upsert(vectors=chunk, namespace="demo")
    print(f"Upserted batch of {len(chunk)} vectors")

time.sleep(5)
print(f"\nFinal stats: {std_index.describe_index_stats()}")

## 3-E: Cleanup

Pinecone's free tier has index limits. Uncomment the cells below when you're done experimenting to free up resources.

In [ ]:
# Uncomment to delete the indexes when finished:

# pc.delete_index(index_name)
# print(f"Deleted index '{index_name}'")

# pc.delete_index(std_index_name)
# print(f"Deleted index '{std_index_name}'")

---

# Key Takeaways

1. **Embeddings are the bridge** between human-readable content and machine-processable numbers. Similar meanings produce similar vectors.

2. **Cosine similarity is king for text** — it measures direction (meaning) while ignoring magnitude. Euclidean distance and dot product are useful in other contexts.

3. **Pinecone is a managed vector database** that handles indexing, scaling, and high-availability so you can focus on your application logic.

4. **Two index patterns**: integrated-embedding indexes (Pinecone embeds for you) and standard indexes (bring your own vectors). Choose based on your need for model control vs. simplicity.

5. **CRUD in Pinecone**: `upsert()` for create/update, `fetch()` for ID lookups, `query()` for similarity search, `delete()` for removal. Always batch large operations.

6. **Metadata filters** let you combine structured queries with semantic search — e.g., "find documents about AI" *and* "category = 'tutorial'".

---

## Preview: Wednesday

Tomorrow we go deeper into the **RAG pipeline**:

- **Text splitting** — chunking documents so they fit inside embedding models
- **Semantic search patterns** — query strategies, top-k tuning, reranking
- **Advanced Pinecone operations** — namespaces, metadata filtering at scale, performance tuning

The vector database you set up today becomes the retrieval engine that feeds context to your LLM chains from last week.